In [ ]:
%pip install yt-dlp faster-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.2/182.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 104.4 MB/s eta 0:00:00


## Configuration

Set `SINGLE_VIDEO_URL` to a YouTube URL to process **only that one video**.  
Leave it as `None` to process **all five videos** in `VIDEO_URLS`.

In [ ]:
# ── Set this to a single URL to process only that video ──────────────────────
# Leave as None to process all five videos in VIDEO_URLS below.
SINGLE_VIDEO_URL = None
# Example: SINGLE_VIDEO_URL = "https://youtu.be/XV-lIaO00H8?si=e4WyccCLpO5Z_7i1"

ALL_VIDEO_URLS = [
    "https://youtu.be/XV-lIaO00H8?si=e4WyccCLpO5Z_7i1",  # video_1
    "https://youtu.be/MZdVAVMgNpA?si=W7CQikonSHMfKrB5",  # video_2
    "https://youtu.be/rWFH6PLOIEI?si=cxIHqdld2qU8vVna",  # video_3
    "https://youtu.be/zVjxEIy33Fs?si=IdVJjdtnMTOycX8m",  # video_4
    "https://youtu.be/ziyiakWUbaQ?si=xZNh8yi7DNM8x42K",  # video_5
]

# Resolve the final list based on the flag above
VIDEO_URLS = [SINGLE_VIDEO_URL] if SINGLE_VIDEO_URL else ALL_VIDEO_URLS
print(f"Will process {len(VIDEO_URLS)} video(s):")
for url in VIDEO_URLS:
    print(f"  • {url}")

In [ ]:
# Cell 2: Download and Transcribe
import os
import gc
from yt_dlp import YoutubeDL
from faster_whisper import WhisperModel

# 1. Define your 5 YouTube URLs here
VIDEO_URLS = [
    "https://youtu.be/XV-lIaO00H8?si=e4WyccCLpO5Z_7i1",
    "https://youtu.be/MZdVAVMgNpA?si=W7CQikonSHMfKrB5",
    "https://youtu.be/rWFH6PLOIEI?si=cxIHqdld2qU8vVna",
    "https://youtu.be/zVjxEIy33Fs?si=IdVJjdtnMTOycX8m",
    "https://youtu.be/ziyiakWUbaQ?si=xZNh8yi7DNM8x42K"
]

def download_audio(url: str, output_name: str) -> str:
    """Downloads the audio stream from a YouTube URL as a .wav file."""
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': f'{output_name}.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'quiet': True
    }
    print(f"Downloading audio for: {url}")
    with YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])
    return f"{output_name}.wav"

def transcribe_audio(audio_path: str, output_txt_path: str) -> None:
    """Transcribes audio using the large-v3 model optimized for T4 GPU."""
    print(f"Loading Whisper large-v3 for {audio_path}...")
    # float16 precision ensures the model fits within the 16GB VRAM of the T4 GPU
    model = WhisperModel("large-v3", device="cuda", compute_type="float16")

    print("Transcribing (this may take a few minutes)...")
    # condition_on_previous_text=False prevents the model from getting stuck in hallucination loops on noisy Hinglish
    segments, info = model.transcribe(audio_path, condition_on_previous_text=False)

    with open(output_txt_path, "w", encoding="utf-8") as f:
        for segment in segments:
            f.write(segment.text + " ")

    print(f"Transcript saved to {output_txt_path}")

    # Free up VRAM for the next iteration
    del model
    gc.collect()

if __name__ == "__main__":
    # Create an output directory
    os.makedirs("transcripts", exist_ok=True)

    for i, url in enumerate(VIDEO_URLS):
        base_name = f"video_{i+1}"
        audio_file = download_audio(url, base_name)
        txt_file = f"transcripts/{base_name}.txt"

        transcribe_audio(audio_file, txt_file)

        # Clean up the heavy .wav file to save disk space
        os.remove(audio_file)
        print(f"Completed processing for {base_name}\n{'-'*40}")

Loading Whisper large-v3 for video_1.wav...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Transcribing (this may take a few minutes)...
Transcript saved to transcripts/video_1.txt
Completed processing for video_1
----------------------------------------


Loading Whisper large-v3 for video_2.wav...
Transcribing (this may take a few minutes)...
Transcript saved to transcripts/video_2.txt
Completed processing for video_2
----------------------------------------


Loading Whisper large-v3 for video_3.wav...
Transcribing (this may take a few minutes)...
Transcript saved to transcripts/video_3.txt
Completed processing for video_3
----------------------------------------


Loading Whisper large-v3 for video_4.wav...
Transcribing (this may take a few minutes)...
Transcript saved to transcripts/video_4.txt
Completed processing for video_4
----------------------------------------


Loading Whisper large-v3 for video_5.wav...
Transcribing (this may take a few minutes)...
Transcript saved to transcripts/video_5.txt
Completed processing for video_5
----------------------------------------
